# H-Optimus-1: Tile Embedding Extraction

**Extract general-purpose histology features from whole-slide H&E images.**

---

## What this notebook does

[H-Optimus-1](https://docs.bioptimus.com/documentation/models/h-optimus) is a
1.1 B-parameter vision transformer (ViT-g/14) trained with self-supervised
learning on billions of tiles from over 1 million slides. It converts each
224×224 tile (at 0.5 µm/px) into a compact **1536-dimensional feature vector**
(embedding) — a numerical fingerprint of tissue morphology.

These embeddings are the input to your own downstream models for tasks such as:

| Downstream task | Approach |
|---|---|
| Mutation prediction | Train a classifier on slide-level aggregated embeddings |
| Survival analysis | Cox regression on embedding features |
| Tissue classification | Cluster or classify tile embeddings |
| Slide retrieval | Nearest-neighbor search in embedding space |

This notebook walks through the **complete pipeline** on a single TCGA-LUAD
slide, from data download to PCA visualization:

1. Download a demo WSI from the GDC portal
2. Build a cohort and configure the inference pipeline
3. Generate tissue masks (skip background tiles)
4. Extract H-Optimus-1 embeddings
5. Inspect output structure and visualize with PCA
6. (Optional) Low-level API for advanced users

---

### Documentation links

| Resource | URL |
|---|---|
| H-Optimus model page | [docs.bioptimus.com/documentation/models/h-optimus](https://docs.bioptimus.com/documentation/models/h-optimus) |
| SDK overview | [docs.bioptimus.com/guides/get-started/sdk](https://docs.bioptimus.com/guides/get-started/sdk) |
| Inference facade | [docs.bioptimus.com/guides/get-started/inference-facade](https://docs.bioptimus.com/guides/get-started/inference-facade) |
| Tile embeddings & PCA guide | [docs.bioptimus.com/guides/workflows/embeddings-pca](https://docs.bioptimus.com/guides/workflows/embeddings-pca) |
| Cohorts guide | [docs.bioptimus.com/guides/workflows/cohort](https://docs.bioptimus.com/guides/workflows/cohort) |
| WSI reader reference | [docs.bioptimus.com/guides/reference/wsi-reader](https://docs.bioptimus.com/guides/reference/wsi-reader) |
| Visualizing results | [docs.bioptimus.com/guides/get-started/visualizing-results](https://docs.bioptimus.com/guides/get-started/visualizing-results) |

### Deployment links

| Resource | URL |
|---|---|
| Deployment — overview & options | [docs.bioptimus.com/deployment/deployment/overview](https://docs.bioptimus.com/deployment/deployment/overview) |
| Deployment — requirements | [docs.bioptimus.com/deployment/deployment/requirements](https://docs.bioptimus.com/deployment/deployment/requirements) |
| Deployment — AWS & SageMaker (`aws` backend) | [docs.bioptimus.com/deployment/platforms/aws-sagemaker](https://docs.bioptimus.com/deployment/platforms/aws-sagemaker) |
| Deployment — on-premise server (`remote` backend) | [docs.bioptimus.com/deployment/platforms/on-premise](https://docs.bioptimus.com/deployment/platforms/on-premise) |


## Prerequisites

| Requirement | Details |
|---|---|
| **Python** | 3.12+ (pre-installed in this environment) |
| **SDK** | `bioptimus-sdk` (installed in the cell below) |
| **Inference backend** | One of: a running Bioptimus server (`remote`), a SageMaker endpoint (`aws`), or local GPU with `.pt2` checkpoints (`local`) |
| **Input data** | Downloaded automatically in Section 1: one `.svs` diagnostic slide from TCGA-LUAD |
| **Disk space** | ~2 GB for the slide + ~200 MB for outputs |
| **Python packages** | `numpy`, `zarr`, `requests`, `scikit-learn`, `matplotlib` (installed below) |

> **Local backend additional requirements:** A CUDA GPU compatible with the
> exported `.pt2` models (default: `sm_86` — A10G, RTX 3090) and
> `pip install bioptimus-sdk[torch]` (installs `torch` and `torchvision`).


## 0. Environment Setup

Quiets noisy startup warnings, verifies the Python version, and installs the
Bioptimus SDK with visualization dependencies.

In [ ]:
# --- Logging / warning configuration ----------------------------------------
# Quiet noisy PyTorch and Python
# warnings unless LOG_LEVEL=DEBUG. Runs before torch / the SDK is imported so
# that TORCH_LOGS and TORCH_CPP_LOG_LEVEL take effect. Export any variable (or
# set LOG_LEVEL=DEBUG) before launching Jupyter to override these defaults.
import os
import warnings

LOG_LEVEL = os.environ.setdefault("LOG_LEVEL", "INFO")

if LOG_LEVEL == "DEBUG":
    os.environ.setdefault("PYTHONWARNINGS", "default")
    os.environ.setdefault("TORCH_LOGS", "+inductor")
    os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "WARNING")
else:
    os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning,ignore::UserWarning")
    os.environ.setdefault("TORCH_LOGS", "-all")
    os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")
    # PYTHONWARNINGS only affects subprocesses and warnings not yet triggered,
    # so also filter the running kernel process.
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import importlib.metadata
import sys

assert sys.version_info >= (3, 12), f"Python 3.12+ is required, but found {sys.version}."
print(f"Python:     {sys.version}")
print(f"Executable: {sys.executable}")

# If the SDK is not already installed, uncomment the line below and run this cell.
# The [torch] extra is REQUIRED for the local (in-process GPU) backend that this
# notebook uses by default; it installs torch and torchvision.
# %pip install -q "bioptimus-sdk[torch]" scikit-learn matplotlib

try:
    sdk_version = importlib.metadata.version("bioptimus-sdk")
    print(f"Bioptimus:  {sdk_version}")
except importlib.metadata.PackageNotFoundError:
    print("Bioptimus:  Not installed — uncomment the %pip line above and re-run this cell.")


### Imports

We import the SDK's high-level building blocks:

| Import | Purpose | Docs |
|---|---|---|
| `Inference` | One-object pipeline facade — tissue masking, embedding, workspace management | [Inference facade](https://docs.bioptimus.com/guides/get-started/inference-facade) |
| `Cohort` | Multi-slide experiment manifest — tracks WSIs, masks, and outputs | [Cohorts](https://docs.bioptimus.com/guides/workflows/cohort) |
| `Models` | Enum of available model names (`H1`, `M_OPTIMUS`, `TISSUE_SEG`, etc.) | [Choosing a model](https://docs.bioptimus.com/documentation/models/choosing-a-model) |
| `Backend` | Enum of inference backends (`REMOTE`, `AWS`, `LOCAL`) | [SDK overview](https://docs.bioptimus.com/guides/get-started/sdk) |
| `bioptimus.utils` | Shared helpers for download, visualization, and output loading | — |


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from bioptimus import utils
from bioptimus.data.cohort import Cohort
from bioptimus.inference import Inference
from bioptimus.models import Backend, Models

---

## 1. Demo Data Setup

We download a single **diagnostic H&E slide** (`.svs`) from the
[GDC Data Portal](https://portal.gdc.cancer.gov/) for TCGA-LUAD sample
**TCGA-75-7027** ([case page](https://portal.gdc.cancer.gov/cases/488646a1-0bdb-45f8-8c30-bef006924511)).

TCGA-LUAD (lung adenocarcinoma) is one of the most widely studied TCGA cohorts
in computational pathology — a good baseline for verifying that the pipeline
produces sensible morphological features.

> **Note:** The download is ~2 GB. Run the next cell once; it skips re-downloading
> if the file already exists.

In [ ]:
# --- Demo data configuration ------------------------------------------------
DEMO_DATA_DIR = Path.cwd() / "demo_external_data"
DEMO_WSI_DIR = DEMO_DATA_DIR / "wsi"

# TCGA-LUAD sample TCGA-75-7027 (diagnostic slide).
# GDC case: https://portal.gdc.cancer.gov/cases/488646a1-0bdb-45f8-8c30-bef006924511
SAMPLE_NAME = "TCGA-75-7027"
SVS_URL = "https://api.gdc.cancer.gov/data/3614fe76-11a3-4b95-a0d6-426ec32fe90c"

DEMO_WSI_DIR.mkdir(parents=True, exist_ok=True)
svs_output_path = DEMO_WSI_DIR / f"{SAMPLE_NAME}.svs"

utils.download_if_needed(SVS_URL, svs_output_path)

print(f"\nDemo WSI directory: {DEMO_WSI_DIR}")

### Sanity check — slide inspection

Before running inference, it is good practice to inspect the slide to confirm
the download succeeded and to understand the data you are working with.

The SDK's [`WSI`](https://docs.bioptimus.com/guides/reference/wsi-reader) factory
auto-selects the fastest available backend (CuCIM if installed, otherwise
OpenSlide) and exposes a uniform API for reading slide metadata, extracting
regions, and generating thumbnails.

Key properties to check:
- **Dimensions** — the full resolution in pixels (typically 50k–100k per side).
- **MPP** (microns per pixel) — the physical resolution of the scan.
- **Levels** — the number of pyramid levels available for multi-resolution reading.

`utils.display_slide_info()` accepts an optional `thumbnail_size=(w, h)` to
control the thumbnail resolution (default `(512, 512)`).

In [ ]:
# Display slide metadata and get a thumbnail image.
wsi_thumbnail = utils.display_slide_info(svs_output_path)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(wsi_thumbnail)
ax.set_title(f"Slide thumbnail — {SAMPLE_NAME}")
ax.axis("off")
plt.tight_layout()
plt.show()

print("\nCompare with the slide image on the GDC case page to confirm correct download.")

---

## 2. Configuration

Configure the inference backend and experiment parameters.

The SDK supports three backends. **This notebook is configured to use the
`local` backend by default** (in-process GPU, no server needed) — see the
code cell below to switch.

| Backend | When to use | Key parameters | Deployment guide |
|---|---|---|---|
| `"remote"` | You have a running Bioptimus FastAPI server (Docker or bare-metal) | `api_url` | [On-premise server](https://docs.bioptimus.com/deployment/platforms/on-premise) |
| `"aws"` | You have a deployed SageMaker endpoint | `endpoint_name`, `region_name` | [AWS & SageMaker](https://docs.bioptimus.com/deployment/platforms/aws-sagemaker) |


> **Not sure which to pick?** If you were given a server URL, use `remote`.
> If you deployed on AWS, use `aws`. If you just have a GPU machine and the
> model checkpoint files, use `local` (the default here).
>
> **Setting up a backend?** Start with the
> [deployment overview](https://docs.bioptimus.com/deployment/deployment/overview)
> to compare options.
>
> **Academic use:** H-Optimus-1 weights are also available directly on
> [Hugging Face](https://docs.bioptimus.com/deployment/platforms/hugging-face)
> under a non-commercial research license (`timm`, no server or AWS account
> required).

The [`Inference`](https://docs.bioptimus.com/guides/get-started/inference-facade)
facade organizes outputs into a structured **workspace**:

```
<output_path>/<experiment>/run_<run>/<variant>/
    config.yaml           # reproducible pipeline config
    tissue/<slide>.png     # cached tissue masks
    h1/embeddings/<slide>.zarr  # embedding outputs
```

> **Tip:** Use the `variant` parameter to compare different setups
> (e.g. `variant="fp32"` vs. `variant="fp16"`) without overwriting results.

> ⚠️ **Before you run the next cell — fill in your settings.**
> The cell below uses the `local` backend, which needs model checkpoint files.
> You **must** set these two paths to real `.pt2` files on disk:
> - `H1_CHECKPOINT` — path to the H-Optimus-1 `.pt2` checkpoint
> - `TISSUE_SEG_CHECKPOINT` — path to the tissue-segmentation `.pt2` checkpoint
>
> The remaining settings (`OUTPUT_DIR`, `EXPERIMENT`, `VARIANT`, `RUN`) come with
> working defaults — keep them as-is for a first run, or change them to organize
> your results. **Where do the checkpoints come from?** They are provided by
> Bioptimus. If
> you don't have them, use the `remote` or `aws` backend instead.


In [ ]:
# --- Experiment configuration -------------------------------------------------
# Choose ONE backend. The `local` backend (Option 3) is ACTIVE BY DEFAULT below.
# To use `remote` or `aws` instead, uncomment that option here AND update the
# matching block in the `Inference(...)` call in Section 4.

# Option 1: Remote server (HTTP).
# API_URL = "http://0.0.0.0:8080"
# utils.check_server(API_URL)

# Option 2: AWS SageMaker — uncomment and set your endpoint:
# ENDPOINT_NAME = "your-h-optimus-endpoint"
# REGION_NAME = "us-east-1"

# Option 3: Local in-process GPU inference (no server required) — ACTIVE DEFAULT.
# Requires .pt2 checkpoints and: pip install "bioptimus-sdk[torch]".
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "The local backend requires a CUDA GPU, but none was found. "
        "Either run on a GPU machine, or switch to the remote/aws backend "
        "(comment out this Option 3 block and uncomment Option 1 or 2)."
    )

num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  cuda:{i} — {torch.cuda.get_device_name(i)}")

# Select which GPU to use (0-indexed). Each process uses a single GPU.
# For multi-GPU parallelism, run separate processes with different GPU_ID values.
GPU_ID = 0
DEVICE = f"cuda:{GPU_ID}"
print(f"\nUsing: {DEVICE} ({torch.cuda.get_device_name(GPU_ID)})")

# --- REQUIRED: set these to real .pt2 files on disk -------------------------
# Provided by Bioptimus.
# The dict keys ("h1", "tissue-seg") are fixed identifiers — do not rename them.
H1_CHECKPOINT = "<PATH TO H1 CHECKPOINT>"
TISSUE_SEG_CHECKPOINT = "<PATH TO TISSUE SEG CHECKPOINT>"
CHECKPOINTS = {
    "h1": str(H1_CHECKPOINT),
    "tissue-seg": str(TISSUE_SEG_CHECKPOINT),
}

# --- Experiment settings (sensible defaults — change them if you like) ------
WSI_DIR = DEMO_WSI_DIR
OUTPUT_DIR = Path("<PATH TO OUTPUT DIRECTORY>")  # where results are written
EXPERIMENT = "<EXPERIMENT NAME>"  # a name for this experiment
VARIANT = "<VARIANT NAME>"  # label for this configuration
RUN = "<RUN NUMBER>"  # a unique identifier for this run to keep results separate
MASK_THRESHOLD = 0.5


---

## 3. Build a Cohort

A [`Cohort`](https://docs.bioptimus.com/guides/workflows/cohort) is the
**single source of truth** for a multi-slide experiment. It discovers WSIs
in a directory, groups them by patient, and tracks each slide's processing
status (masks, embeddings, predictions) across models.

[`Cohort.from_directories()`](https://docs.bioptimus.com/guides/workflows/cohort)
recursively scans `wsi_dir` for supported slide formats (`.svs`, `.tiff`,
`.ndpi`, `.mrxs`, etc.) and creates a `WSIRecord` for each file found.

Other options for building or managing cohorts:
- `Cohort.from_directories(wsi_dir, bulk_rna_dir=..., mask_dir=..., patient_id_fn=...)`
  — auto-pair WSIs with RNA and masks.
- `Cohort.from_csv(csv_path)` — build from a CSV manifest.
- `cohort.save("manifest.yaml")` / `Cohort.load("manifest.yaml")` — persist
  and reload a cohort.

In [ ]:
cohort = Cohort.from_directories(wsi_dir=WSI_DIR)

print(cohort.summary())
print(f"\nWSI IDs:    {cohort.wsi_ids}")
print(f"Patients:   {cohort.patient_ids}")
print(f"Modalities: {cohort[0].available_modalities}")

---

## 4. Create the Inference Pipeline

The [`Inference`](https://docs.bioptimus.com/guides/get-started/inference-facade)
facade replaces the manual multi-step pipeline (Backbone → mask provider →
SlideInference → writer) with a **single, reusable object**.

Key constructor parameters:

| Parameter | Value | Purpose |
|---|---|---|
| `model_name` | `Models.H1` | Selects H-Optimus-1 (1536-d embeddings) |
| `cohort` | our cohort | Slides to process |
| `tissue` | `True` | Enable automatic tissue masking |
| `mask_threshold` | `0.5` | Keep tiles with ≥50% tissue |
| `workers` | `1` | Number of slides processed concurrently |

Additional optional parameters:
- `output_format="zarr"` — also supports `"hdf5"` and `"npz"`.
- `max_concurrency=256` — max concurrent tile requests to the server.
- `wsi_backend="cucim"` — force a specific WSI reader (`"cucim"`,
  `"openslide"`, `"tifffile"`).
- `description="..."` — free-text description saved in the workspace config.

> **Resume a previous run:** `Inference.from_workspace("path/to/workspace")`
> reloads the config and cohort manifest from a saved workspace.

> **Backend selection:** The code cell below uses the `local` backend.
> See the commented sections to switch to `remote` (HTTP server) or
> `aws` (SageMaker). See the
> [local backend tutorial](../docs/sdk-tutorial-local.md) for full details.


In [ ]:
# --- Create the Inference pipeline ---------------------------------------------
# The `local` backend block is ACTIVE BY DEFAULT below. To use remote or aws,
# comment out the "Option 3: Local" lines and uncomment the matching block
# (and set its variables in Section 2). Only ONE backend may be active at a time.

infer = Inference(
    model_name=Models.H1,
    # --- Option 1: Remote server (HTTP) ---
    # backend=Backend.REMOTE,
    # api_url=API_URL,
    # timeout=120.0,
    # --- Option 2: AWS SageMaker ---
    # backend=Backend.AWS,
    # endpoint_name=ENDPOINT_NAME,
    # region_name=REGION_NAME,
    # timeout=120.0,
    # --- Option 3: Local in-process GPU (no server) — ACTIVE DEFAULT ---
    backend=Backend.LOCAL,
    checkpoints=CHECKPOINTS,
    device=DEVICE,
    # --- Common parameters (all backends) ---
    cohort=cohort,
    tissue=True,
    mask_threshold=MASK_THRESHOLD,
    output_path=OUTPUT_DIR,
    experiment=EXPERIMENT,
    variant=VARIANT,
    run=RUN,
    workers=1,
)

print(f"Model:     {infer.model_name}")
print(f"Workspace: {infer.workspace}")
print(f"WSIs:      {len(cohort)}")


---

## 5. Tissue Mask Generation

Whole-slide images are mostly empty background. Running the model on every tile
would be wasteful — **tissue masking** identifies which tiles contain tissue
and skips the rest.

### How it works

1. The slide is tiled at a **coarse resolution** (512×512 at 8 µm/px).
2. The bundled [tissue segmentation model](https://docs.bioptimus.com/documentation/models/tissue-segmentation)
   classifies each coarse tile as tissue or background.
3. A binary mask is saved to disk and **reused** for all subsequent models.
4. Only tiles above `mask_threshold` tissue fraction are processed.

On a typical TCGA slide, tissue masking reduces the number of tiles from
~16,000 to ~2,000–5,000, cutting inference time by 3–8×.

> **Caching:** Masks are saved to `<workspace>/tissue/<slide>.png`. Re-running
> `infer.tissue()` automatically skips slides that already have masks.
> Pass `force=True` to recompute, or a single `wsi_path` to process just
> one slide.

In [ ]:
# Generate tissue masks.
infer.tissue()

print(cohort.summary())

In [ ]:
# Visualize the tissue mask alongside the slide thumbnail.
first_record = cohort[0]

utils.plot_slide_and_mask(
    wsi_path=first_record.wsi_path,
    mask_path=first_record.mask_path,
    threshold=MASK_THRESHOLD,
)

# Print mask statistics using the same threshold as the pipeline.
mask = plt.imread(str(first_record.mask_path))
if mask.ndim == 3:
    mask = mask[:, :, 0]
mask_binary = mask >= MASK_THRESHOLD
print(f"Mask shape:   {mask_binary.shape} (rows × cols of tiles)")
print(f"Tissue tiles: {mask_binary.sum():,} / {mask_binary.size:,} ({mask_binary.mean():.1%})")

---

## 6. Extract H-Optimus-1 Embeddings

Now we run the main inference step: for every tissue tile (224×224 at
0.5 µm/px), the server returns a **1536-dimensional embedding vector**.

The [`Inference.run(mode="embed")`](https://docs.bioptimus.com/guides/get-started/inference-facade)
method:

1. Tiles each slide at the model's native resolution (0.5 µm/px).
2. Filters tiles using the pre-computed tissue mask.
3. Dispatches tiles concurrently to the server.
4. Streams results to a Zarr store on disk (constant memory).

Outputs are written to `<workspace>/h1/embeddings/<slide>.zarr`.

> **Resumability:** If the process is interrupted, re-running `infer.run()`
> automatically picks up where it left off. Pass `force=True` to reprocess
> all slides regardless of existing outputs.

> **Troubleshooting:** After a run, `infer.failures` returns a list of any
> WSIs that encountered errors. Use `infer.status()` to print an overview
> of cohort processing progress.

In [ ]:
# Extract H-Optimus-1 embeddings.
infer.run(mode="embed")

print(cohort.summary())
infer.report()

---

## 7. Inspect the Output

Inference outputs are stored as [Zarr](https://zarr.readthedocs.io/) directory
stores. This format supports memory-mapped access and streaming writes,
making it ideal for large slides.

`utils.load_zarr_output()` returns a dict with all available arrays and
metadata:

| Key | Shape | Description |
|---|---|---|
| `"outputs"` | `(n_tiles, 1536)` | Tile embeddings |
| `"coords"` | `(n_tiles, 2)` | `(x, y)` pixel coordinates of each tile |
| `"metadata"` | dict | Slide attributes (`slide_name`, `tile_size`, `mpp`, etc.) |
| `"tissue_ratios"` | `(n_tiles,)` | Tissue fraction per tile (if stored) |
| `"thumbnail"` | `(H, W, 3)` | Slide thumbnail for overlay visualization (if stored) |
| `"tissue_mask"` | `(H, W)` | Tissue mask used during extraction (if stored) |
| `"gene_names"` | list | Output gene list — M-Optimus predictions only |

See [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
for the full schema.

In [ ]:
# Load the output.
first_record = cohort[0]
h1_output = first_record.outputs.get(infer.model_name)
embedding_path = h1_output.embedding_path

print(f"Output path: {embedding_path}")
print(f"Modalities:  {h1_output.modalities}")
print()

data = utils.load_zarr_output(embedding_path)
embeddings = data["outputs"]
coords = data["coords"]
metadata = data["metadata"]

print(f"Embeddings shape: {embeddings.shape}  (tiles × embedding_dim)")
print(f"Coords shape:     {coords.shape}  (tiles × 2)")
print(f"Embedding dtype:  {embeddings.dtype}")
print()

print("Metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

---

## 8. PCA Analysis — Morphological Structure

Principal Component Analysis (PCA) is a quick way to check that embeddings
capture meaningful biological variation. The first few components typically
correspond to tissue vs. background (or dominant tissue type), tumor vs.
stroma, or immune-rich vs. immune-poor regions.

`utils.plot_pca_scatter()` options:
- `n_components` — number of PCA components to compute (default `2`).
- `cmap` — any matplotlib colormap (`"viridis"`, `"plasma"`, `"coolwarm"`,
  `"tab10"`, etc.).
- `point_size` — scatter point size (default `10`).
- `figsize` — figure dimensions in inches (default `(8, 6)`).

See the [Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
guide for more details on interpreting these components.

In [ ]:
# 2D PCA scatter — tiles colored by index (proxy for spatial location).
scores, pca = utils.plot_pca_scatter(
    embeddings,
    n_components=3,
    title=f"H-Optimus-1 Embeddings PCA — {first_record.wsi_id}",
    cmap="viridis",
)

### Spatial PCA — Mapping Components Back to the Slide

The scatter plot above shows structure in embedding space, but we can also
**project PC scores back onto the slide coordinates** to see what each
component captures spatially.

This is a powerful QC tool: if PC1 cleanly separates tumor from stroma,
the model is extracting biologically relevant features. See
[Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
for more visualization techniques.

`utils.plot_spatial_pca()` options:
- `components` — which PCs to plot, 0-indexed (default: first 3).
- `figsize` — figure dimensions (default `(18, 5)`).

In [ ]:
# Spatial PCA heatmaps — one panel per principal component.
utils.plot_spatial_pca(
    scores,
    coords,
    components=[0, 1, 2],
    title_prefix=f"{first_record.wsi_id} — ",
)

### Visual Proof — Nearest-Neighbor Tiles

PCA shows structure, but do tiles with **similar embeddings actually look
alike**? This is the ultimate test of embedding quality.

For each query tile, we find its closest neighbors in cosine distance and
display them side-by-side. If the model captures meaningful morphology:

- Tumor tiles should neighbor other tumor tiles
- Stromal tiles should neighbor other stromal tiles
- Immune-rich tiles should cluster together

The query tiles are sampled from different PCA regions so we see a range of
tissue types. See [Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
for details on interpreting embedding neighborhoods.

`utils.plot_nearest_neighbor_tiles()` options:
- `query_indices` — pick specific tile indices instead of auto-sampling.
- `n_queries` — number of auto-sampled queries (default `4`).
- `n_neighbors` — neighbors per query (default `5`).
- `seed` — random seed for reproducible query selection (default `42`).
- `figsize_per_tile` — figure size per tile in inches (default `1.8`).

In [ ]:
# Nearest-neighbor tiles in embedding space.
utils.plot_nearest_neighbor_tiles(
    embeddings,
    coords,
    wsi_path=first_record.wsi_path,
    n_queries=4,
    n_neighbors=5,
    tile_size=224,
    mpp=0.5,
)

---

## 9. (Optional) Low-Level API

The `Inference` facade is convenient for most use cases, but the SDK also
exposes a **lower-level API** for users who need more control.

The two-layer architecture:

| Layer | Class | Use case |
|---|---|---|
| **High-level** | `Inference` | Cohort-level pipelines with caching, workspaces, and resume |
| **Low-level** | `Backbone` + `SlideInference` | Single-slide control, custom mask providers, custom writers |

The low-level API lets you:
- Choose the WSI backend explicitly (`CuCIM`, `OpenSlide`, `TiffFile`)
- Provide your own tissue mask (a NumPy array)
- Control the output format (`ZARR`, `HDF5`, `NPZ`)
- Adjust `max_concurrency` per-slide

See the [SDK overview](https://docs.bioptimus.com/guides/get-started/sdk) and
[Tile embeddings & PCA](https://docs.bioptimus.com/guides/workflows/embeddings-pca)
for full examples.

In [ ]:
# --- Low-level API example ---------------------------------------------------
# This example uses the LOCAL backend (matching the rest of the notebook).
# To use remote or aws instead, comment out the "Local" lines and uncomment the
# matching backend lines in steps 1 and 2.

from bioptimus.models.backbones import Backbone
from bioptimus.preprocess.wsi.models.bioptimus_mask import BioptimusTissueMaskModel
from bioptimus.preprocess.wsi.provider.tiled import TiledTissueMask
from bioptimus.inference.inference import SlideInference
from bioptimus.inference.writers import OutputFormat

# 1. Create H1 model client.

# --- Remote (HTTP server): ---
# model = Backbone(Models.H1, backend=Backend.REMOTE, base_url=API_URL)

# --- SageMaker: ---
# model = Backbone(Models.H1, backend=Backend.AWS,
#                  endpoint_name=ENDPOINT_NAME, region_name=REGION_NAME)

# --- Local (in-process GPU, no server): ---
model = Backbone(
    Models.H1,
    backend=Backend.LOCAL,
    checkpoint=H1_CHECKPOINT,
    device=DEVICE,
)

# 2. Create tissue mask provider.
# Tissue-seg is bundled with the same SageMaker endpoint (no extra cost)
# and the same HTTP server — use the matching backend.

# --- Remote: ---
# tissue_backbone = Backbone(Models.TISSUE_SEG, backend=Backend.REMOTE, base_url=API_URL)

# --- SageMaker (same endpoint as H1): ---
# tissue_backbone = Backbone(Models.TISSUE_SEG, backend=Backend.AWS,
#                            endpoint_name=ENDPOINT_NAME, region_name=REGION_NAME)

# --- Local: ---
tissue_backbone = Backbone(
    Models.TISSUE_SEG,
    backend=Backend.LOCAL,
    checkpoint=TISSUE_SEG_CHECKPOINT,
    device=DEVICE,
)

mask_provider = TiledTissueMask(model=BioptimusTissueMaskModel(backbone=tissue_backbone))

# 3. Run single-slide inference.
inferrer = SlideInference(
    wsi_path=str(svs_output_path),
    model=model,
    mask_provider=mask_provider,
    mask_threshold=0.5,
)
result_path = inferrer.predict(
    output_path="/tmp/h1_lowlevel.zarr",
    output_format=OutputFormat.ZARR,
    max_concurrency=64,  # value shown is for the local backend; use 256 for remote/aws
    mode="embed",
)
print(f"Low-level output: {result_path}")



### Further reading

- [M-Optimus tutorial](./m_optimus_1_tutorial.ipynb) — spatial transcriptomics
  from histology, with and without bulk RNA
- [Choosing a model](https://docs.bioptimus.com/documentation/models/choosing-a-model)
  — H-Optimus vs. M-Optimus comparison
- [Cohorts guide](https://docs.bioptimus.com/guides/workflows/cohort)
  — multi-slide experiments with shared masks
- [Visualizing results](https://docs.bioptimus.com/guides/get-started/visualizing-results)
  — tile-grid heatmaps and gene overlays
- [Benchmarks](https://docs.bioptimus.com/documentation/models/benchmarks)
  — H-Optimus-1 performance on PathBench and HEST